# EXP11 — Protótipo offline de triagem

Este notebook faz parte do pacote de reprodução. Leia as explicações antes de executar. **Candidatos não verificados permanecem sem rótulo confirmado.**

## 1. Preparar o ambiente

No Colab, a primeira célula baixa uma cópia do projeto caso ela ainda não esteja disponível. No Python local, use o ambiente e as dependências descritos no README.

In [ ]:
from pathlib import Path
import os, subprocess, sys

def preparar_projeto():
    atual = Path.cwd()
    for pasta in [atual, *atual.parents]:
        if (pasta / "src" / "ethmix" / "__init__.py").exists():
            raiz = pasta
            break
    else:
        raiz = Path("/content/ethereum-mixer-reproducibility")
        if not raiz.exists():
            subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/BorgesGusta/ethereum-mixer-reproducibility.git",
                str(raiz)], check=True)
    sys.path.insert(0, str(raiz / "src"))
    os.chdir(raiz)
    return raiz

RAIZ = preparar_projeto()
print("Projeto:", RAIZ)


### Bibliotecas

Esta célula instala as bibliotecas necessárias na sessão atual, caso ainda não estejam presentes.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

## 2. O que o protótipo faz?

Lê um exemplo JSON com endereço, score opcional e referências fornecidas por um operador. **Não consulta a blockchain e não bloqueia endereços.** A decisão automática é sempre indeterminada até revisão humana.

In [ ]:
import json
import sys
sys.path.insert(0, str(RAIZ / "security_triage"))
from triage import triage, append_audit, verify_audit

casos = json.loads((RAIZ / "security_triage" / "sample_cases.json").read_text())
resultados = [triage(caso) for caso in casos]
for resultado in resultados:
    print(resultado["contract"], resultado["status"], resultado["priority"])

## 3. Verificar a trilha de auditoria

Os hashes ajudam a detectar modificações locais em registros, desde que o histórico original ou seu hash final esteja preservado. Não impedem alguém de substituir todo o arquivo e recalcular seus hashes.

In [ ]:
from datetime import datetime, timezone
saida = RAIZ / "results" / "generated" / "EXP11"
saida.mkdir(parents=True, exist_ok=True)
log = saida / (datetime.now(timezone.utc).strftime("auditoria_%Y%m%d_%H%M%S.jsonl"))
append_audit(log, "sample_cases.json", resultados)
print(verify_audit(log))
print("Arquivo:", log)